# Diploma: Semantic Segmentation of Aerial Imagery

**Goal:** Compare 5 semantic segmentation architectures on ISPRS Vaihingen and Potsdam datasets, and evaluate whether attention-based U-Net delivers measurable improvements (mIoU, Boundary IoU) within a constrained FLOPs budget.

**Models compared:**
1. **FCN** — baseline fully convolutional network
2. **U-Net** — primary baseline (encoder-decoder with skip connections)
3. **DeepLabV3+** — atrous convolutions / ASPP
4. **Attention U-Net** — U-Net + attention gates (the candidate)
5. **SegFormer** — transformer encoder + lightweight MLP decoder

**Hypotheses:**
- H1: Attention U-Net improves mIoU by >= 5 pp vs U-Net
- H2: Attention U-Net stays within 120% of U-Net FLOPs
- H3: Attention U-Net improves Boundary IoU by >= 15 pp vs U-Net

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
assert torch.cuda.is_available(), "GPU not available!"

In [ ]:
!git clone https://github.com/YOUR_USERNAME/diploma.git
%cd diploma
!pip install -r requirements_colab.txt -q

## Datasets

### 1. ISPRS Vaihingen
Download from: https://www.isprs.org/education/benchmarks/UrbanSemLab/2d-sem-label-vaihingen.aspx
Requires registration. You will receive `ISPRS_semantic_labeling_Vaihingen.zip`.

### 2. ISPRS Potsdam
Download from: https://www.isprs.org/education/benchmarks/UrbanSemLab/2d-sem-label-potsdam.aspx
You will receive `ISPRS_semantic_labeling_Potsdam.zip`.

### 3. Upload to Colab
Either drag-and-drop the zips into the Files panel, or place them in `MyDrive/diploma_data/` and copy:
```bash
!mkdir -p data/raw
!cp /content/drive/MyDrive/diploma_data/*.zip data/raw/
!cd data/raw && unzip -q ISPRS_semantic_labeling_Vaihingen.zip -d vaihingen
!cd data/raw && unzip -q ISPRS_semantic_labeling_Potsdam.zip -d potsdam
```

### Expected folder structure
```
data/raw/
  vaihingen/
    top/                # *.tif RGB tiles
    gts_for_participants/   # *.tif label tiles
  potsdam/
    2_Ortho_RGB/
    5_Labels_all/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/diploma_results"
!mkdir -p {SAVE_DIR}

In [ ]:
!python3 -c "
from code.data.eda import run_full_eda
run_full_eda('data/raw', 'experiments/eda')
"
!python3 -c "
from code.data.preprocessor import DataPreprocessor
for ds in ['vaihingen', 'potsdam']:
    p = DataPreprocessor(ds, 'data/raw', 'data/processed')
    p.run_all()
    print(f'{ds} preprocessing done')
"

In [ ]:
import json
stats = json.load(open('experiments/eda/dataset_stats.json'))
for ds, info in stats.items():
    print(f"{ds}: train={info['actual_train_tiles']} "
          f"val={info['actual_val_tiles']} "
          f"test={info['actual_test_tiles']}")

In [ ]:
import subprocess, json, os
models = ['fcn', 'unet', 'deeplab', 'attention', 'segformer']
for model in models:
    print(f"\n{'='*50}")
    print(f"Training {model}...")
    result = subprocess.run([
        'python3', 'code/training/train.py',
        '--model', model,
        '--dataset', 'vaihingen',
        '--dataset_path', 'data/processed',
        '--epochs', '100',
        '--batch_size', '4',
        '--lr', '1e-4',
        '--save_dir', f'experiments/{model}'
    ], capture_output=True, text=True)
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])

In [ ]:
!python3 code/evaluation/evaluate.py \
    --compare_all \
    --results_dir experiments/ \
    --models fcn unet deeplab attention segformer \
    --dataset vaihingen \
    --dataset_path data/processed \
    --save_dir experiments/comparison

In [ ]:
import json
results = json.load(open('experiments/comparison/compare_results.json'))
print(f"{'Model':<12} {'mIoU':>8} {'BIoU':>8} {'Params_M':>10} {'FLOPs_G':>10}")
print("-" * 52)
for model, metrics in results.items():
    print(f"{model:<12} {metrics['miou']:>8.4f} "
          f"{metrics['boundary_iou']:>8.4f} "
          f"{metrics['params_M']:>10.1f} "
          f"{metrics['flops_G']:>10.1f}")

In [ ]:
baseline_miou = results['unet']['miou']
baseline_flops = results['unet']['flops_G']
baseline_biou = results['unet']['boundary_iou']
attn = results['attention']
print("H1 (mIoU +5pp):", "CONFIRMED" if attn['miou'] >= baseline_miou + 0.05 else "REJECTED",
      f"delta={attn['miou']-baseline_miou:.4f}")
print("H2 (FLOPs +20%):", "CONFIRMED" if attn['flops_G'] <= baseline_flops*1.2 else "REJECTED",
      f"ratio={attn['flops_G']/baseline_flops:.2f}")
print("H3 (BIoU +15pp):", "CONFIRMED" if attn['boundary_iou'] >= baseline_biou+0.15 else "REJECTED",
      f"delta={attn['boundary_iou']-baseline_biou:.4f}")

In [ ]:
import shutil
shutil.copytree('experiments', f'{SAVE_DIR}/experiments', dirs_exist_ok=True)
print(f"Results saved to {SAVE_DIR}")